## ดึงข้อมูล fundamental

In [1]:
import pandas as pd
import yfinance as yf
import psycopg2
from psycopg2.extras import execute_values
import time
import warnings
from IPython.display import display, Markdown

warnings.filterwarnings('ignore')

display(Markdown("### 📊 กำลังดึงข้อมูลปัจจัยพื้นฐานแบบเจาะลึก (Full Fundamentals)"))

try:
    # 1. เชื่อมต่อ Database
    conn = psycopg2.connect(host="127.0.0.1", database="set_quant", user="postgres", password="Shifa.326459")
    cur = conn.cursor()

    # 2. สร้างตารางเก็บงบการเงินและอัตราส่วน (ลบของเก่าทิ้งเพื่ออัปเดตค่าล่าสุด)
    cur.execute("DROP TABLE IF EXISTS fact_fundamentals_full CASCADE;")
    
    create_table_sql = """
    CREATE TABLE fact_fundamentals_full (
        symbol VARCHAR(20) PRIMARY KEY,
        -- A. Balance Sheet (งบดุล) --
        total_assets BIGINT,
        total_debt BIGINT,
        debt_to_equity FLOAT,
        -- B. Income Statement (งบกำไรขาดทุน) --
        total_revenue BIGINT,
        net_income BIGINT,
        eps FLOAT,
        profit_margin FLOAT,
        -- C. Valuation & Benefits (สิทธิประโยชน์และมูลค่า) --
        pe_ratio FLOAT,
        pbv_ratio FLOAT,
        dividend_yield FLOAT,
        payout_ratio FLOAT,
        
        updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );
    """
    cur.execute(create_table_sql)
    conn.commit()
    print("✅ สร้างตาราง 'fact_fundamentals_full' โครงสร้างใหม่สำเร็จ!")

    # 3. ดึงรายชื่อหุ้น
    cur.execute("SELECT symbol FROM set_companies_list")
    symbols = [row[0] for row in cur.fetchall()]
    print(f"⏳ เตรียมดึงงบการเงินของหุ้น {len(symbols)} ตัว (ใช้เวลาประมาณ 5-10 นาที)...\n")

    insert_sql = """
        INSERT INTO fact_fundamentals_full 
        (symbol, total_assets, total_debt, debt_to_equity, total_revenue, net_income, eps, profit_margin, pe_ratio, pbv_ratio, dividend_yield, payout_ratio)
        VALUES %s
        ON CONFLICT (symbol) DO NOTHING;
    """

    success_count = 0
    records = []

    # 4. วนลูปดึงข้อมูล
    for sym in symbols:
        try:
            yf_symbol = f"{sym}.BK"
            ticker = yf.Ticker(yf_symbol)
            info = ticker.info
            
            if not info or 'symbol' not in info:
                continue
                
            # แปลงข้อมูลจาก dictionary ให้ตรงกับตาราง (ดักจับตัวที่ไม่มีค่าให้เป็น None)
            records.append((
                sym,
                # A. งบดุล (บางตัว yfinance เก็บเป็น totalDebt แทน Liabilities รวม)
                info.get('totalAssets', None), 
                info.get('totalDebt', None),
                info.get('debtToEquity', None),
                # B. งบกำไรขาดทุน
                info.get('totalRevenue', None),
                info.get('netIncomeToCommon', None),
                info.get('trailingEps', None),
                info.get('profitMargins', None),
                # C. มูลค่าและปันผล
                info.get('trailingPE', None),
                info.get('priceToBook', None),
                info.get('dividendYield', None),
                info.get('payoutRatio', None)
            ))
            
            success_count += 1
            
            # บันทึกทีละ 50 ตัว
            if len(records) >= 50:
                execute_values(cur, insert_sql, records)
                conn.commit()
                records = [] 
                
            time.sleep(0.3) # พักเครื่อง

        except Exception as e:
            print(f"⚠️ ข้ามหุ้น {sym} (Error: {e})")
            continue

    # บันทึกส่วนที่เหลือ
    if records:
        execute_values(cur, insert_sql, records)
        conn.commit()

    print(f"\n🎉 ALL DONE! ดึงข้อมูลครบเซ็ตสำเร็จ {success_count} / {len(symbols)} ตัว")

    # 5. โชว์ผลลัพธ์การดึงข้อมูล
    display(Markdown("#### 🔎 ตัวอย่างข้อมูลงบการเงิน (สุ่มมา 5 ตัว)"))
    df_show = pd.read_sql("SELECT * FROM fact_fundamentals_full WHERE pe_ratio IS NOT NULL LIMIT 5;", conn)
    display(df_show)

except Exception as e:
    print(f"❌ Database Error: {e}")

### 📊 กำลังดึงข้อมูลปัจจัยพื้นฐานแบบเจาะลึก (Full Fundamentals)

✅ สร้างตาราง 'fact_fundamentals_full' โครงสร้างใหม่สำเร็จ!
⏳ เตรียมดึงงบการเงินของหุ้น 925 ตัว (ใช้เวลาประมาณ 5-10 นาที)...



HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: WELL.BK"}}}



🎉 ALL DONE! ดึงข้อมูลครบเซ็ตสำเร็จ 924 / 925 ตัว


#### 🔎 ตัวอย่างข้อมูลงบการเงิน (สุ่มมา 5 ตัว)

,symbol,total_assets,total_debt,debt_to_equity,total_revenue,net_income,eps,profit_margin,pe_ratio,pbv_ratio,dividend_yield,payout_ratio,updated_at
0,2S,None,10209000,0.500,7432499200,149699008,0.27,0.02014,9.555555,0.694295,9.02,0.4444,2026-03-03 00:20:17.090401
1,88TH,None,12305000,2.667,610267904,99107968,0.42,0.16240,12.380952,2.395210,NaN,0.0000,2026-03-03 00:20:17.090401
2,A5,None,1777730048,105.653,1314095360,102945408,0.06,0.07834,30.333334,1.308411,2.73,1.6667,2026-03-03 00:20:17.090401
3,AAI,None,176110624,3.654,7104616960,740689600,0.38,0.10425,10.263158,1.712780,6.46,1.2453,2026-03-03 00:20:17.090401
4,AAV,None,43000119296,317.196,49092370432,2336219904,0.08,0.04759,14.375000,1.090048,NaN,0.0000,2026-03-03 00:20:17.090401


## ดึงข้อมูลการจ่ายปันผลย้อนหลัง

In [5]:
import pandas as pd
import yfinance as yf
import psycopg2
from psycopg2.extras import execute_values
import time
import warnings
from IPython.display import display, Markdown

warnings.filterwarnings('ignore')

display(Markdown("### 💰 กำลังดึงประวัติการจ่ายเงินปันผล "))

try:
    # 1. เชื่อมต่อ Database
    conn = psycopg2.connect(host="127.0.0.1", database="set_quant", user="postgres", password="Shifa.326459")
    cur = conn.cursor()

    # 2. ล้างข้อมูลเก่า
    cur.execute("TRUNCATE TABLE fact_dividends;")
    conn.commit()
    
    insert_sql = """
        INSERT INTO fact_dividends (symbol, ex_date, dividend_amount)
        VALUES %s;
    """

    # 3. ดึงรายชื่อหุ้น
    cur.execute("SELECT symbol FROM set_companies_list")
    symbols = [row[0] for row in cur.fetchall()]
    print(f"⏳ เตรียมเช็คประวัติปันผลหุ้น {len(symbols)} ตัว (ดึง 10 ปี อาจใช้เวลาสักครู่)...\n")

    # ⭐️ เปลี่ยนเป็นดึงข้อมูลย้อนหลัง 10 ปี
    ten_years_ago = pd.Timestamp.now(tz='UTC') - pd.DateOffset(years=10)

    success_count = 0
    total_dividend_records = 0
    records = []

    # 4. วนลูปดึงข้อมูล
    for sym in symbols:
        try:
            yf_symbol = f"{sym}.BK"
            ticker = yf.Ticker(yf_symbol)
            
            divs = ticker.dividends
            
            if divs.empty:
                continue
                
            divs.index = pd.to_datetime(divs.index, utc=True)
            divs_10y = divs[divs.index >= ten_years_ago]
            
            if divs_10y.empty:
                continue

            for date, amount in divs_10y.items():
                if pd.isna(amount) or amount <= 0:
                    continue
                    
                records.append((sym, date.date(), float(amount)))
                total_dividend_records += 1
            
            success_count += 1
            
            if len(records) >= 50:
                execute_values(cur, insert_sql, records)
                conn.commit()
                records = []
                
            time.sleep(0.1) # พักเครื่องเบาๆ

        except Exception as e:
            # ⭐️ จุดสำคัญ: เมื่อเจอ Error ต้องสั่ง Rollback เพื่อปลดล็อค Database แล้วให้ไปตัวถัดไป
            conn.rollback()
            # print(f"⚠️ ข้ามหุ้น {sym} (Error: {e})") 
            continue

    # บันทึกส่วนที่เหลือ
    if records:
        execute_values(cur, insert_sql, records)
        conn.commit()

    print(f"\n🎉 ALL DONE! ดึงประวัติปันผล 10 ปีสำเร็จ พบหุ้นที่จ่ายปันผล {success_count} ตัว")
    print(f"📊 ได้ข้อมูลการจ่ายเงินปันผลรวมทั้งสิ้น {total_dividend_records:,} ครั้ง ในรอบ 10 ปี")

    # 5. โชว์ผลลัพธ์
    display(Markdown("#### 🔎 ตัวอย่างประวัติการจ่ายเงินปันผล 10 ปี (เรียงล่าสุด)"))
    df_show = pd.read_sql("SELECT * FROM fact_dividends ORDER BY ex_date DESC LIMIT 5;", conn)
    display(df_show)

except Exception as e:
    print(f"❌ Database Error: {e}")

### 💰 กำลังดึงประวัติการจ่ายเงินปันผล 

⏳ เตรียมเช็คประวัติปันผลหุ้น 925 ตัว (ดึง 10 ปี อาจใช้เวลาสักครู่)...



HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: WELL.BK"}}}
$WELL.BK: possibly delisted; no timezone found



🎉 ALL DONE! ดึงประวัติปันผล 10 ปีสำเร็จ พบหุ้นที่จ่ายปันผล 829 ตัว
📊 ได้ข้อมูลการจ่ายเงินปันผลรวมทั้งสิ้น 8,341 ครั้ง ในรอบ 10 ปี


#### 🔎 ตัวอย่างประวัติการจ่ายเงินปันผล 10 ปี (เรียงล่าสุด)

,symbol,ex_date,dividend_amount
0,LEE,2026-03-25,0.2000
1,INETREIT,2026-03-25,0.0675
2,ACE,2026-03-25,0.0100
3,RAM,2026-03-24,0.1000
4,KCE,2026-03-22,0.6000


In [20]:
import pandas as pd
import psycopg2
from IPython.display import display, Markdown

try:
    # เชื่อมต่อ Database
    conn = psycopg2.connect(host="127.0.0.1", database="set_quant", user="postgres", password="Shifa.326459")
    
    # ==========================================
    # 💰 Part 1: Total Dividend (ข้อ 5.29 - 5.30)
    # ==========================================
    query_total = """
        SELECT 
            symbol, 
            SUM(dividend_amount) as total_dividend
        FROM fact_dividends
        WHERE ex_date >= CURRENT_DATE - INTERVAL '5 years'
        GROUP BY symbol
        ORDER BY total_dividend DESC;
    """
    df_total = pd.read_sql(query_total, conn)
    
    # 🎯 5.29
    ans_5_29 = len(df_total)
    display(Markdown(f"### 5.29 which stocks is given dividend last 5 years"))
    display(Markdown(f"พบหุ้นที่จ่ายเงินปันผลในช่วง 5 ปีที่ผ่านมา จำนวนทั้งสิ้น **{ans_5_29}** บริษัท"))
    
    # 🎯 5.30
    display(Markdown(f"### 5.30 list stock is given dividend last 5 years from max to min"))
    
    df_530 = df_total.head(20).copy()
    df_530['total_dividend'] = df_530['total_dividend'].apply(lambda x: f"{x:,.4f} ฿")
    df_530.columns = ['Symbol', 'Total Dividend (5 Years)']
    display(df_530)

    display(Markdown("---"))

    # ==========================================
    # 📈 Part 2: Growth Dividend (ข้อ 5.31 - 5.32)
    # ==========================================
    query_yearly = """
        SELECT 
            symbol, 
            EXTRACT(YEAR FROM ex_date) as pay_year, 
            SUM(dividend_amount) as yearly_div
        FROM fact_dividends
        WHERE ex_date >= CURRENT_DATE - INTERVAL '5 years'
        GROUP BY symbol, EXTRACT(YEAR FROM ex_date)
    """
    df_yearly = pd.read_sql(query_yearly, conn)
    df_pivot = df_yearly.pivot(index='symbol', columns='pay_year', values='yearly_div').fillna(0)
    
    years = sorted(df_pivot.columns.tolist())
    start_y, end_y = years[0], years[-1]
    
    # คำนวณคะแนน และ Overall Growth
    df_pivot['growth_score'] = 0
    for i in range(len(years) - 1):
        df_pivot['growth_score'] += (df_pivot[years[i+1]] > df_pivot[years[i]]).astype(int)
        
    df_pivot['overall_pct'] = df_pivot.apply(
        lambda row: ((row[end_y] - row[start_y]) / row[start_y] * 100) if row[start_y] > 0 else 0, 
        axis=1
    )
    
    # กรองและจัดเรียง (เอา Top 20)
    df_filtered = df_pivot[(df_pivot['growth_score'] > 0) & (df_pivot['overall_pct'] > 0)].copy()
    df_filtered = df_filtered.sort_values(['growth_score', 'overall_pct'], ascending=[False, False]).head(20)
    df_filtered = df_filtered.reset_index()

    # 🎯 5.31
    ans_5_31 = len(df_filtered) 
    
    total_growth_stocks = len(df_pivot[(df_pivot['growth_score'] > 0) & (df_pivot['overall_pct'] > 0)])
    
    display(Markdown(f"### 5.31 which stocks is given growth dividend last 5 years"))
    display(Markdown(f"พบหุ้นที่มีการเติบโตของเงินปันผล (Growth Dividend) จำนวนทั้งสิ้น **{total_growth_stocks}** บริษัท"))

    # 🎯 5.32
    display(Markdown(f"### 5.32 list stock is given growth dividend last 5 years from max to min"))
    
    # --- ตาราง 1 (DPS) ---
    df_amount = pd.DataFrame()
    df_amount['Symbol'] = df_filtered['symbol']
    df_amount['Growth Score'] = df_filtered['growth_score'].apply(lambda x: f"{x}/{len(years)-1}")
    
    for y in years:
        df_amount[f"{int(y)} (Baht)"] = df_filtered[y].apply(lambda x: f"{x:.4f}" if x > 0 else "-")

    display(Markdown("**Table 1: Yearly Dividend Amount (Baht/Share)**"))
    display(df_amount)

    # --- ตาราง 2 (YoY %) ---
    df_percent = pd.DataFrame()
    df_percent['Symbol'] = df_filtered['symbol']
    df_percent['Overall Growth'] = df_filtered['overall_pct'].apply(lambda x: f"{x:+.2f}%")
    
    for i in range(1, len(years)):
        prev_y = years[i-1]
        curr_y = years[i]
        col_name = f"{int(prev_y)} -> {int(curr_y)} (%)"
        
        def calc_yoy(row):
            old = row[prev_y]
            new = row[curr_y]
            if old == 0 and new > 0:
                return "N/A (New)"
            elif old == 0 and new == 0:
                return "-"
            else:
                pct = ((new - old) / old) * 100
                return f"{pct:+.2f}%"
                
        df_percent[col_name] = df_filtered.apply(calc_yoy, axis=1).values

    def color_growth(val):
        if isinstance(val, str):
            if val.startswith('-') and len(val) > 1:
                return 'color: #D32F2F; font-weight: bold;'
            elif val.startswith('+'):
                return 'color: #388E3C; font-weight: bold;'
        return ''

    cols_to_color = df_percent.columns[1:]
    
    try:
        styled_df = df_percent.style.map(color_growth, subset=cols_to_color)
    except AttributeError:
        styled_df = df_percent.style.applymap(color_growth, subset=cols_to_color)

    display(Markdown("**Table 2: Year-over-Year (YoY) Growth Percentage**"))
    display(styled_df)

except Exception as e:
    print(f"❌ Error: {e}")
finally:
    if 'conn' in locals() and conn:
        conn.close()

### 5.29 which stocks is given dividend last 5 years

พบหุ้นที่จ่ายเงินปันผลในช่วง 5 ปีที่ผ่านมา จำนวนทั้งสิ้น **766** บริษัท

### 5.30 list stock is given dividend last 5 years from max to min

,Symbol,Total Dividend (5 Years)
0,KYE,85.5500 ฿
1,METCO,80.0000 ฿
2,STANLY,74.0000 ฿
3,ADVANC,68.9000 ฿
4,KWC,52.0000 ฿
5,ALUCON,50.0000 ฿
6,SCC,48.5000 ฿
7,SCCC,47.0000 ฿
8,PTTEP,42.1250 ฿
9,TISCO,38.7000 ฿


---

...

### 5.32 list stock is given growth dividend last 5 years from max to min

**Table 1: Yearly Dividend Amount (Baht/Share)**

,Symbol,Growth Score,2021 (Baht),2022 (Baht),2023 (Baht),2024 (Baht),2025 (Baht),2026 (Baht)
0,ADVANC,5/5,3.4500,7.6900,8.2400,9.4800,12.6300,27.4100
1,TNL,4/5,0.0697,0.4980,0.2000,0.3000,0.4000,0.6000
2,BH,4/5,1.1500,3.2000,3.7000,5.1500,5.0000,9.0000
3,CCET,4/5,0.0169,0.0270,0.0491,0.1450,0.2000,0.0900
4,TTT,4/5,0.5000,1.0000,1.2500,2.1500,2.8000,1.5000
5,BIZ,4/5,0.1667,0.4278,0.2000,0.3000,0.3500,0.5000
6,BDMS,4/5,0.2500,0.5000,0.6500,0.7000,0.7500,0.6500
7,MC,4/5,0.2000,0.6000,0.8100,0.9000,0.9600,0.5200
8,LHSC,4/5,0.0996,0.3787,0.8372,1.0495,1.1450,0.2410
9,PT,4/5,0.2500,0.6100,0.6500,0.7000,1.2000,0.5000


**Table 2: Year-over-Year (YoY) Growth Percentage**

,Symbol,Overall Growth,2021 -> 2022 (%),2022 -> 2023 (%),2023 -> 2024 (%),2024 -> 2025 (%),2025 -> 2026 (%)
0,ADVANC,+694.49%,+122.90%,+7.15%,+15.05%,+33.23%,+117.02%
1,TNL,+760.83%,+614.49%,-59.84%,+50.00%,+33.33%,+50.00%
2,BH,+682.61%,+178.26%,+15.62%,+39.19%,-2.91%,+80.00%
3,CCET,+432.54%,+59.76%,+81.85%,+195.32%,+37.93%,-55.00%
4,TTT,+200.00%,+100.00%,+25.00%,+72.00%,+30.23%,-46.43%
5,BIZ,+199.94%,+156.63%,-53.25%,+50.00%,+16.67%,+42.86%
6,BDMS,+160.00%,+100.00%,+30.00%,+7.69%,+7.14%,-13.33%
7,MC,+160.00%,+200.00%,+35.00%,+11.11%,+6.67%,-45.83%
8,LHSC,+141.97%,+280.22%,+121.07%,+25.36%,+9.10%,-78.95%
9,PT,+100.00%,+144.00%,+6.56%,+7.69%,+71.43%,-58.33%
